# 06 BIP-Entwicklung als animierte Bubble-Chart (Rosling-Stil)
Dieses Notebook visualisiert die BIP-Entwicklung der Länder als animierte Blasen-Chart im Stil von **Hans Rosling / Gapminder**: eine Blase je Land, Größe = Bevölkerung, Farbe = Region, Play-Button animiert über die Jahre.

Es ist ein **eigenständiges Visualisierungs-Notebook** (nicht Teil der Modell-Kette 01–05): Es nutzt dieselben `src`-Module und dieselbe World-Bank-Datenquelle, baut sich aber ein eigenes *kontemporäres* Panel, weil die Animation die Ist-Werte je Jahr braucht (das Modell-Panel enthält bewusst nur gelaggte Treiber).

**Achsen:** x = BIP pro Kopf (log), y = Anteil der 15–64-Jährigen, Größe = Bevölkerung. Beide Wahlmöglichkeiten (x-Achse und Länderumfang) sind unten per Umschalter einstellbar.

## 0 Setup


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config, data_utils, plotting

print("Projekt-Wurzel:", ROOT)
print("Bevölkerung im Indikator-Set:",
      "population" in {n for n, _ in config.INDICATORS.values()})

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Projekt-Wurzel: c:\Users\antoi\Documents\repos\AI-for-Business-Prognosis\CapstoneProjekt_BPI
Bevölkerung im Indikator-Set: True


## 1 Umschalter: hier die Darstellung wählen

Zwei Entscheidungen, jeweils eine Zeile:

* **`X_ACHSE`** — `"gdp_pc"` (BIP pro Kopf, Rosling-Standard) oder `"gdp_total"` (absolutes BIP = BIP pro Kopf × Bevölkerung).
* **`NUR_MODELL_LAENDER`** — `True`: dieselben 116 Länder wie die Modelle (Projekt-konsistent). `False`: alle Länder mit brauchbarer Historie (vollere Optik).

In [12]:
X_ACHSE = "gdp_pc"          # "gdp_pc"  oder  "gdp_total"
NUR_MODELL_LAENDER = True   # True = 116 Modell-Länder, False = alle
JAHRE = (1990, config.LAST_YEAR)   # Animationszeitraum (Abdeckung ab 1990 hoch)

## 2 Daten laden (inkl. Bevölkerung)

`download_wdi()` erkennt automatisch, dass mit `SP.POP.TOTL` ein neuer Indikator dazugekommen ist, und lädt die Rohdaten bei Bedarf neu (der alte Cache ohne Bevölkerung wird verworfen).

In [13]:
raw = data_utils.download_wdi()
static = data_utils.download_country_meta()

assert "population" in raw["indicator"].unique(), \
    "Bevölkerung fehlt in den Rohdaten - bitte download_wdi(force=True) ausführen."
print("Rohdaten:", raw.shape, "| Indikatoren:", sorted(raw['indicator'].unique()))

Cache-Treffer: C:\Users\antoi\Documents\repos\AI-for-Business-Prognosis\CapstoneProjekt_BPI\data\raw\wdi_raw_long.csv (force=True zum Neuladen)
Rohdaten: (126945, 4) | Indikatoren: ['dependency_ratio', 'gdp_growth', 'gdp_pc', 'inflation', 'investment_gdp', 'pop_growth', 'population', 'trade_gdp', 'working_age_share']


## 3 Kontemporäres Viz-Panel bauen

`build_viz_panel()` baut — anders als das Modell-Panel — ein Panel mit **Ist-Werten je Jahr** (keine Lags): BIP pro Kopf, Erwerbsanteil und Bevölkerung, angereichert um Region und Ländername.

In [14]:
viz = data_utils.build_viz_panel(
    raw, static, only_modeling_countries=NUR_MODELL_LAENDER)

# Absolutes BIP (falls gewählt) = BIP pro Kopf x Bevölkerung
viz["gdp_total"] = viz["gdp_pc"] * viz["population"]

print(f"Viz-Panel: {viz.shape} | {viz['country'].nunique()} Länder | "
      f"Jahre {viz['year'].min()}-{viz['year'].max()}")
viz[["country", "name", "region", "year", "gdp_pc",
     "working_age_share", "population"]].head()

Viz-Panel: (7540, 15) | 116 Länder | Jahre 1960-2024


,country,name,region,year,gdp_pc,working_age_share,population
0,ABW,Aruba,Latin America & Caribbean,1960,NaN,54.632024,54922.0
1,ABW,Aruba,Latin America & Caribbean,1961,NaN,54.953804,55578.0
2,ABW,Aruba,Latin America & Caribbean,1962,NaN,55.230824,56320.0
3,ABW,Aruba,Latin America & Caribbean,1963,NaN,55.618712,57002.0
4,ABW,Aruba,Latin America & Caribbean,1964,NaN,56.131658,57619.0


## 4 Datencheck vor der Animation

Kurz prüfen, wie viele Länder je Jahr alle drei Achsen-Größen (x, y, Größe) besetzt haben — das sind die Blasen, die pro Frame erscheinen.

In [15]:
x_col = X_ACHSE
frame_counts = (viz[(viz["year"].between(*JAHRE))]
                .dropna(subset=[x_col, "working_age_share", "population"])
                .groupby("year")["country"].nunique())
print(f"Blasen je Jahr (min/median/max): {frame_counts.min()} / "
      f"{int(frame_counts.median())} / {frame_counts.max()}")
frame_counts.tail(6)

Blasen je Jahr (min/median/max): 116 / 116 / 116


year
2019    116
2020    116
2021    116
2022    116
2023    116
2024    116
Name: country, dtype: int64

## 5 Die Animation

Play drücken — oder den Jahres-Slider ziehen. Blasen bewegen sich nach rechts (steigendes BIP pro Kopf) und meist nach oben/unten je nach demografischer Entwicklung. Die Achsen sind über alle Jahre fixiert, damit die Bewegung sichtbar wird.

In [16]:
titel = ("BIP-Entwicklung der Länder — "
         + ("BIP pro Kopf" if X_ACHSE == "gdp_pc" else "absolutes BIP")
         + " vs. Erwerbsanteil, Blasengröße = Bevölkerung")

fig = plotting.animate_gapminder(
    viz, x=X_ACHSE, y="working_age_share", size="population",
    color="region", year_range=JAHRE, log_x=True, title=titel)
fig.show()

## 6 Export als interaktive HTML

Die Animation als eigenständige HTML-Datei ausgeben.

In [17]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_html = config.RESULTS_DIR / "bip_animation.html"
fig.write_html(str(out_html), include_plotlyjs="cdn", auto_play=False)
print("Gespeichert:", out_html)

Gespeichert: C:\Users\antoi\Documents\repos\AI-for-Business-Prognosis\CapstoneProjekt_BPI\data\results\bip_animation.html
